In [1]:
import numpy as np
import pandas as pd


S0 = 50.0
sigma_true = 0.30   
q = 0.04            
r = 0.04            

T_list = np.array([1/12, 1/4, 1/2, 1.0, 2.0], dtype=float)
C_m_list = np.array([1.72, 2.96, 4.14, 5.73, 7.75], dtype=float)

S_disc = S0 * np.exp(-q * T_list)

ratio = C_m_list / S_disc

# Safety check: ratio must be < 1 for logs in approx 2/3 to make sense
if np.any(ratio >= 1.0):
    raise ValueError("Cm / (S0 e^{-qT}) must be < 1 for approx 2 and 3.")


# (1) sigma_imp,approx,1 = sqrt(2pi/T) * Cm/(S0 e^{-qT})
sigma1 = np.sqrt(2*np.pi / T_list) * ratio

# (2) sigma_imp,approx,2 = sqrt( - (2pi/T) * ln( 1 - (Cm/(S0 e^{-qT}))^2 ) )
sigma2 = np.sqrt(-(2*np.pi / T_list) * np.log(1 - ratio**2))

# (3) sigma_imp,approx,3 = sqrt( - (4/T) * sqrt(8/pi) * ln( 1 - (Cm/(S0 e^{-qT}))^2 ) )
sigma3 = np.sqrt(-(4 / T_list) * np.sqrt(8/np.pi) * np.log(1 - ratio**2))

# Relative errors 
# |sigma - sigma_approx| / sigma_approx
err1 = np.abs(sigma_true - sigma1) / sigma1
err2 = np.abs(sigma_true - sigma2) / sigma2
err3 = np.abs(sigma_true - sigma3) / sigma3

# Build the output table 

col_names = ["T=1/12", "T=1/4", "T=1/2", "T=1", "T=2"]

table = pd.DataFrame(
    index=[
        "|σ-σ_imp,approx,1| / σ_imp,approx,1",
        "|σ-σ_imp,approx,2| / σ_imp,approx,2",
        "|σ-σ_imp,approx,3| / σ_imp,approx,3",
        "σ_imp,approx,1",
        "σ_imp,approx,2",
        "σ_imp,approx,3",
    ],
    columns=col_names
)

table.loc["|σ-σ_imp,approx,1| / σ_imp,approx,1"] = err1
table.loc["|σ-σ_imp,approx,2| / σ_imp,approx,2"] = err2
table.loc["|σ-σ_imp,approx,3| / σ_imp,approx,3"] = err3
table.loc["σ_imp,approx,1"] = sigma1
table.loc["σ_imp,approx,2"] = sigma2
table.loc["σ_imp,approx,3"] = sigma3

pd.set_option("display.float_format", lambda x: f"{x:0.6f}")
print(table)

                                      T=1/12    T=1/4    T=1/2      T=1  \
|σ-σ_imp,approx,1| / σ_imp,approx,1 0.001001 0.000776 0.001844 0.003402   
|σ-σ_imp,approx,2| / σ_imp,approx,2 0.000703 0.000120 0.000053 0.000182   
|σ-σ_imp,approx,3| / σ_imp,approx,3 0.007158 0.007975 0.007803 0.008036   
σ_imp,approx,1                      0.299700 0.299768 0.299448 0.298983   
σ_imp,approx,2                      0.299789 0.300036 0.299984 0.300055   
σ_imp,approx,3                      0.302163 0.302412 0.302359 0.302430   

                                         T=2  
|σ-σ_imp,approx,1| / σ_imp,approx,1 0.008024  
|σ-σ_imp,approx,2| / σ_imp,approx,2 0.000860  
|σ-σ_imp,approx,3| / σ_imp,approx,3 0.007002  
σ_imp,approx,1                      0.297612  
σ_imp,approx,2                      0.299742  
σ_imp,approx,3                      0.302115  
